# Import libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Data preprocessing

## Reading the data

In [ ]:
data = pd.read_csv("info_updated.csv", sep=";")
data

## Cleaning

### Copy of data

In [ ]:
# Make a copy of the current dataset
data_copy = data.copy()
data_copy

### Null values

In [ ]:
data_copy.isnull().sum()

### Duplicated

In [ ]:
data_copy.duplicated().sum()

### "Bad" Data

In [ ]:
bad_data = data_copy[data_copy.eq('---').any(axis=1)]
bad_data

In [ ]:
data_copy.eq("---").sum().loc[lambda s: s > 0]

#### ISS Reception
We will work with data received with 99% or more ISS Reception.


In [ ]:
data_copy[data_copy["ISS Recept"] <= 99]

<center>

|   ISS Recept    |  Count   |
|:---------------:|:--------:|
|   $\leq 10\%$   |   188    |
|   $\leq 20\%$   |   201    |
|   $\leq 25\%$   |   209    |
|   $\leq 30\%$   |   214    |
|   $\leq 40\%$   |   244    |
|   $\leq 50\%$   |   1789   |
|   $\leq 60\%$   |   2396   |
|   $\leq 70\%$   |   3126   |
|   $\leq 75\%$   |   3648   |
|   $\leq 80\%$   |   4067   |
|   $\leq 90\%$   |   5018   |
|   $\leq 95\%$   |   5910   |
| **$\leq 99\%$** | **6710** |

</center>

6710 records were obtained with an ISS Reception inferior to 99\%. We want to work with the most reliable data, so we will drop those records.

In [ ]:
data_copy = data_copy[data_copy["ISS Recept"] >= 99]
data_copy

In [ ]:
data_copy.eq("---").sum().loc[lambda s: s > 0]

#### Wind Related Measurements


In [ ]:
wind_headers = ["Wind Speed", "Wind Direction", "Wind Run", "Hi Speed", "Hi Dir"]
data_copy[wind_headers]

##### No wind detected at all during the interval
Wind Speed, Wind Run & Hi Speed = 0

In [ ]:
no_wind_at_all = (data_copy["Wind Speed"] == 0) & (data_copy["Wind Run"] == 0) & (data_copy["Hi Speed"] == 0)
data_copy[no_wind_at_all]

In [ ]:
data_copy.loc[no_wind_at_all, ["Wind Direction", "Hi Dir"]] = "No Wind"
data_copy

In [ ]:
data_copy.eq("---").sum().loc[lambda s: s > 0]

##### Only gusts
Wind Speed & Wind Run = 0, Hi Speed > 0

In [ ]:
only_gusts = (data_copy["Wind Speed"] == 0) & (data_copy["Wind Run"] == 0) & (data_copy["Hi Speed"] > 0) & (data_copy["Wind Direction"] == "---")
data_copy[only_gusts]

In [ ]:
data_copy.loc[only_gusts, ["Wind Direction"]] = "Calm with gusts"
data_copy

In [ ]:
data_copy.eq("---").sum().loc[lambda s: s > 0]

#### Removal of UV, "Feel Like" Indexes and Solar related measurements

1. UV Columns: Not needed for this project.
2. Indexes: Qualitative descriptions (approximate a feeling).
3. Solar Columns: Not needed for this project.

In [ ]:
data_copy = data_copy.drop(["UV Index", "UV Dose", "Hi UV", "Wind Chill", "Heat Index", "THW Index", "THSW Index", "In Heat", "Solar Rad", "Solar Energy", "Hi Solar Rad"], axis=1)
data_copy

In [ ]:
data_copy.eq("---").sum().loc[lambda s: s > 0]

We managed to deal with all the bad data present in the original dataset.

## Data Conversion

### Date (ISO 8601)

In [ ]:
correct_data = data_copy.copy()
correct_data["Date"] = pd.to_datetime(correct_data["Date"] + correct_data["Time"], format="%d-%m-%y%H:%M")
correct_data = correct_data.drop(["Time"], axis=1)
correct_data

### Floats

In [ ]:
to_float_headers = ["Avg Temp", "Hi Temp", "Low Temp", "Dew Pt", "Wind Speed", "Wind Run", "Hi Speed", "Bar", "Heat D-D", "Cool D-D", "In Temp", "In Dew", "In EMC", "In Air Density"]
correct_data[to_float_headers] = correct_data[to_float_headers].astype(float)
correct_data

### Integers

In [ ]:
to_int_headers = ["Out Hum", "In Hum"]
correct_data[to_int_headers] = correct_data[to_int_headers].astype(int)
correct_data

### Strings

In [ ]:
to_str_headers = ["Wind Direction", "Hi Dir"]
correct_data[to_str_headers] = correct_data[to_str_headers].astype('string')
correct_data

## Downloading the new dataset

In [ ]:
correct_data.to_csv("dataset.csv", index=False, sep=";")

## Extra: Setting Date as Index
2008-01-11: Arc Int changed from 1min to 10min

2008-01-18: Arc Int changed from 10min to 15min

In [ ]:
correct_data = correct_data.set_index("Date")

In [ ]:
for year in range(2009, 2026):
    row_amount = len(correct_data.loc[str(year)])
    print(f"{year}: {row_amount} of {96*365} expected records")

## Analysis: Year 2025

In [ ]:
data_2025 = correct_data.loc["2025"]
data_2025

<center>
<h1>2025 Temperatures</h1>

|      Month      |    Average     |      High      |          Low           |
|:---------------:|:--------------:|:--------------:|:----------------------:|
|     January     |  $23^\circ C$  | $35.3^\circ C$ |     $9.8^\circ C$      |
|    February     | $22.5^\circ C$ | $38.3^\circ C$ |      $7^\circ C$       |
|      March      | $19.2^\circ C$ | $36.5^\circ C$ |     $4.4^\circ C$      |
|      April      | $15.4^\circ C$ | $32.4^\circ C$ |     $3.7^\circ C$      |
|       May       | $12.3^\circ C$ | $30.1^\circ C$ |     $0.2^\circ C$      |
|      June       | $8.3^\circ C$  | $26.6^\circ C$ |     $-2.8^\circ C$     |
|      July       | $9.6^\circ C$  | $27.1^\circ C$ |     $-2.5^\circ C$     |
|     August      | $10.5^\circ C$ | $28.5^\circ C$ |     $-1.7^\circ C$     |
|    September    |  $13^\circ C$  | $31.4^\circ C$ |     $0.1^\circ C$      |
|     October     | $16.9^\circ C$ | $32.6^\circ C$ |     $0.5^\circ C$      |
|    November     | $18.7^\circ C$ | $34.1^\circ C$ |     $4.8^\circ C$      |
|    December     | $21.4^\circ C$ | $36.3^\circ C$ |     $7.7^\circ C$      |

<h4>Highest: $38.3^\circ C$ (February 8th, 17:30)</h4>
<h4>Lowest: $-2.8^\circ C$ (June 30th, 08:00-08:15)</h4>
</center>

In [ ]:
correct_data